# Chapter 06 — Resilient Operations, Disaster Recovery & Rolling Updates

Welcome to the interactive notebook for **Chapter 06**!

> 📖 **Concept Reference**: For graph compatibility rules during in-place `--update` and zero-disruption disaster recovery seek strategies, see [Chapter 06 README](README.md).

### Interactive Workflow
1. Inspect historical replay telemetry records.
2. Execute the local rolling update simulation comparing Pipeline V1 vs. V2.
3. Verify transform name stability and backward-compatible schema evolution.
4. Review the Pub/Sub subscription seek disaster recovery workflow.

## 1. Environment Setup & Kernel Selection

This project uses **[`uv`](https://docs.astral.sh/uv/)** to manage dependencies in an isolated `.venv` virtual environment.

> 📘 **Full Environment Setup Guide**: For complete setup instructions, terminal commands, and kernel registration, see the root [README.md](../README.md) (*Section: Environment Setup & Activating the Kernel with uv*).

* **Option A: In the Terminal**: Activate with `source .venv/bin/activate` or execute commands directly with `uv run`.
* **Option B: In This Jupyter Notebook (Kernel Selection)**:
  If you encounter `ModuleNotFoundError: No module named 'apache_beam'`, register the kernel from your terminal:
  ```bash
  uv run python -m ipykernel install --user --name dataflow-workshops --display-name "Python (dataflow-workshops)"
  ```
  Then click the **Kernel Picker** in the top-right corner of this notebook editor and select **`Python (dataflow-workshops)`** (or browse to `.venv/bin/python`).

Run the verification cell below to confirm your setup:

In [1]:
import os
import sys

print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

Python Executable: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python3
✅ Success: Notebook is running inside the project's .venv!


✅ Apache Beam Version: 2.77.0


## 2. Architecture & Visual DAG

Here is the architecture for this chapter, illustrating rolling updates and Pub/Sub disaster recovery seek replay:

![Chapter 06 Resilient Operations & Updates DAG](../artifacts/06_resilient_ops_and_updates_dag.png)

<details>
<summary><b>Click to expand Mermaid source code</b></summary>

```mermaid
flowchart LR
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef update fill:#f3e8fd,stroke:#9334e6,stroke-width:2px,color:#6b21a8,font-weight:bold;
    classDef dlq fill:#fee2e2,stroke:#ef4444,stroke-width:2px,color:#991b1b,font-weight:bold;

    subgraph RollingUpdate["1. Zero-Downtime Rolling Update"]
        JOB_V1["Running Job V1<br/>Standard Tax (5%)"]:::ptrans
        CHECK{"Compatibility Check<br/>--update"}:::update
        JOB_V2["Running Job V2<br/>Loyalty Discount (10%)"]:::update
        JOB_V1 --> CHECK --> JOB_V2
    end

    subgraph PubSub["2. Pub/Sub 7-Day Retention & Seek"]
        TOPIC[("Pub/Sub Topic<br/>orders-stream")]:::pcoll
        PROD_SUB[("Live Sub<br/>orders-stream-sub")]:::pcoll
        RECOV_SUB[("Recovery Sub<br/>seek to -24h")]:::update
        DLQ[("Dead-Letter Topic<br/>max_delivery=5")]:::dlq
        TOPIC --> PROD_SUB
        TOPIC --> RECOV_SUB
        PROD_SUB -. "Poison Pills" .-> DLQ
    end

    subgraph Sinks["3. Zero-Disruption Backfill"]
        PROD_BQ[("BigQuery: processed_orders")]:::sink
        TEMP_BQ[("BigQuery: temp_backfill")]:::update
        SWAP["Atomic Partition Swap"]:::ptrans
        JOB_V2 --> PROD_BQ
        RECOV_SUB --> TEMP_BQ --> SWAP --> PROD_BQ
    end
```
</details>

## 3. Inspecting Historical Replay Telemetry

Let's inspect the historical telemetry dataset (`data/replay_telemetry.json`) simulating 24 hours of order transactions:

In [2]:
import json
from pathlib import Path

records = []
with open("data/replay_telemetry.json") as f:
    for i, line in enumerate(f):
        if i < 4:
            records.append(json.loads(line))

print("Sample Historical Replay Records:")
for r in records:
    print(json.dumps(r, indent=2))

Sample Historical Replay Records:
{
  "order_id": "ord-00001",
  "customer_id": "cust_015",
  "subtotal": 291.29,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:19:22.076349+00:00"
}
{
  "order_id": "ord-00002",
  "customer_id": "cust_012",
  "subtotal": 117.34,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:29:22.076349+00:00"
}
{
  "order_id": "ord-00003",
  "customer_id": "cust_007",
  "subtotal": 174.02,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:39:22.076349+00:00"
}
{
  "order_id": "ord-00004",
  "customer_id": "cust_011",
  "subtotal": 292.4,
  "tax_rate": 0.05,
  "order_timestamp": "2026-10-26T19:49:22.076349+00:00"
}


## 4. Transform Name Stability & In-Place Rolling Updates

Inspect how preserving transform labels enables Dataflow to map state seamlessly during `--update`.
*(For the 4 graph compatibility rules, see [Chapter 06 README](README.md)).*

In [3]:
import inspect
from pipelines import resilient_stream_v1, resilient_stream_v2

print("=== V1 Calculate Taxes Logic ===")
print(inspect.getsource(resilient_stream_v1.CalculateTaxV1Fn))

print("=== V2 Calculate Taxes Logic (with Loyalty Discount) ===")
print(inspect.getsource(resilient_stream_v2.CalculateTaxV2Fn))

=== V1 Calculate Taxes Logic ===
class CalculateTaxV1Fn(beam.DoFn):
    """V1 Tax Processing: standard 5% sales tax."""

    def process(self, order: Dict[str, Any]) -> Iterator[Dict[str, Any]]:
        subtotal = float(order.get("subtotal", 0.0))
        tax = round(subtotal * 0.05, 2)
        total = round(subtotal + tax, 2)
        now_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()

        yield {
            "order_id": str(order.get("order_id")),
            "customer_id": str(order.get("customer_id")),
            "subtotal": subtotal,
            "tax_amount": tax,
            "total_amount": total,
            "pipeline_version": "v1.0.0",
            "processed_at": now_iso,
        }

=== V2 Calculate Taxes Logic (with Loyalty Discount) ===
class CalculateTaxV2Fn(beam.DoFn):
    """V2 Upgraded Tax Processing: includes 10% loyalty discount for orders >= $100."""

    def process(self, order: Dict[str, Any]) -> Iterator[Dict[str, Any]]:
        subtotal = float

## 5. Disaster Recovery via Pub/Sub Subscription Seek

Replay historical offsets using dedicated recovery subscriptions without interrupting production consumers.
*(For the staging table and atomic partition swap workflow, see [Chapter 06 README](README.md)).*

## 6. Running the Local Rolling Update Simulation

Let's execute `scripts/01_test_rolling_update_simulation.sh` to run both V1 and V2 pipelines locally:

In [4]:
import subprocess

cmd = ["bash", "scripts/01_test_rolling_update_simulation.sh"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

 Chapter 06: Simulating Rolling Pipeline Update (V1 -> V2)
Input Telemetry: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/data/replay_telemetry.json
V1 Output:       /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/output/orders_v1*.json
V2 Output:       /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/output/orders_v2*.json
----------------------------------------------------------------------
[1/2] Running Pipeline Version 1 (Standard 5% Tax)...
[2/2] Running Pipeline Version 2 (10% Loyalty Discount on >=$100)...
----------------------------------------------------------------------
 Both versions executed successfully! Generated files in /Users/zelda.luconi/Medium/google-products/dataflow-workshops/06_operations_replay_and_updates/output:
total 96
drwxr-xr-x@ 4 zelda.luconi  staff    128 Oct 27 20:25 .
drwxr-xr-x@ 8 zelda.luconi  staff    256 O

## 7. Inspecting Output Records: V1 vs. V2 Schema Evolution

Notice how V2 seamlessly introduced `loyalty_discount` and `net_amount` while preserving backward compatibility:

In [5]:
import glob
import json

def print_sample(pattern, label):
    print(f"=== {label} ===")
    files = sorted(glob.glob(pattern))
    if files:
        with open(files[0]) as f:
            for line in f.readlines()[:3]:
                print(json.dumps(json.loads(line.strip()), indent=2))
    print()

print_sample("output/orders_v1*.json", "Version 1 Records")
print_sample("output/orders_v2*.json", "Version 2 Records (with Loyalty Discount)")

=== Version 1 Records ===
{
  "order_id": "ord-00001",
  "customer_id": "cust_015",
  "subtotal": 291.29,
  "tax_amount": 14.56,
  "total_amount": 305.85,
  "pipeline_version": "v1.0.0",
  "processed_at": "2026-10-27T19:25:50.412418+00:00"
}
{
  "order_id": "ord-00002",
  "customer_id": "cust_012",
  "subtotal": 117.34,
  "tax_amount": 5.87,
  "total_amount": 123.21,
  "pipeline_version": "v1.0.0",
  "processed_at": "2026-10-27T19:25:50.413738+00:00"
}
{
  "order_id": "ord-00003",
  "customer_id": "cust_007",
  "subtotal": 174.02,
  "tax_amount": 8.7,
  "total_amount": 182.72,
  "pipeline_version": "v1.0.0",
  "processed_at": "2026-10-27T19:25:50.413789+00:00"
}

=== Version 2 Records (with Loyalty Discount) ===
{
  "order_id": "ord-00001",
  "customer_id": "cust_015",
  "subtotal": 291.29,
  "loyalty_discount": 29.13,
  "net_amount": 262.16,
  "tax_amount": 13.11,
  "total_amount": 275.27,
  "pipeline_version": "v2.0.0",
  "processed_at": "2026-10-27T19:25:53.883120+00:00"
}
{
  "orde

## 8. Summary & Next Steps

### What You Learned in Chapter 06:
1. **In-Place Zero-Downtime Updates**: Using `--update` to deploy new code without stopping pipelines or risking duplicate processing.
2. **Graph Compatibility**: Maintaining stable transform labels and schema backward compatibility.
3. **Pub/Sub Subscription Seek**: Backfilling historical data using dedicated recovery subscriptions without interrupting production consumers.
4. **Poison Pill Defense**: Using Pub/Sub Dead-Letter Topics with maximum delivery attempt limits.

### Deploying to Google Cloud Dataflow:
To deploy V1 and perform an in-place update on GCP, run:

```bash
bash scripts/02_deploy_v1_and_update_v2.sh
```

In **Chapter 07**, we will explore **Unit Testing, Docker Flex Templates, CI/CD & Security** (`TestPipeline`, `PAssert`, `TestStream`, Artifact Registry Flex Templates, CMEK, and Private IPs).